Importing and converting "CRIME" data to .csv to work with it

In [ ]:
pip install pyaxis

CHANGING DIRECTORY SO THAT WE'RE IN THE RIGHT PLACE

In [ ]:

%cd "/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/DATA"
%pwd

Reading .px file

In [ ]:
from pyaxis import pyaxis

data = pyaxis.parse("Data_Crime_messy", encoding="utf-8-sig") #reading .PX file --> "encoding=..." necessary bc required by pyaxis

In [ ]:
data
#df = data['DATA'] #creating dataframe out of dictionary so that we can actually work with the data inside DATA folder

#df.to_csv('Crime_clean.csv', index=False) #index = False --> so that not 0, 1, .... row lables are created



In [ ]:
import pandas as pd

#crimeClean = pd.read_csv("Crime_clean.csv")

In [ ]:
#crimeClean.columns #looking if it imported everything correctly

Importing and converting socioeconomic data to .csv to work with it

In [ ]:
#pop = pyaxis.parse("Data_Population_messy", encoding="utf-8-sig")

#pop
#dfp = pop['DATA'] #creating dataframe out of dictionary so that we can actually work with the data inside DATA folder

#dfp.to_csv('Population_clean.parquet', index=False)

#PopClean = pd.read_parquet("Population_clean.parquet")

#PopClean.columns

Trying with Parquet bc .csv was too huge

In [ ]:
import sys
!{sys.executable} -m pip install pyarrow #pip command is not working and I cannot fix--> workaround

In [ ]:
from pyaxis import pyaxis 
import pandas as pd
import pyarrow


pop = pyaxis.parse("Data_Population_messy", encoding="utf-8-sig")

pop
dfp = pop['DATA'] #creating dataframe out of dictionary so that we can actually work with the data inside DATA folder

dfp.to_parquet('Population_clean.parquet', index=False)

PopClean = pd.read_parquet("Population_clean.parquet")

print(PopClean.columns)
print(PopClean.head())

Dropping unnecessary variables so that the file gets smaller

In [ ]:
#PopClean = PopClean.drop(columns=['Kanton (-)', 'Gemeinde (......)'])

In [ ]:
#print(PopClean[PopClean['Kanton (-) / Bezirk (>>) / Gemeinde (......)'].str.contains('Wettswil', na=False)].head())

values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #show me unique values to see how I need to adjust it
print(values)

In [ ]:
#dropping all Gemeinde and Bezirk data (granular level we do not need)

PopClean = PopClean[~PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.contains(r">>|\.\.\.\.\.\.", na=False)]
values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique()
print(values)

#"~" inverts boolean condition, so that PopClean contains only rows that do not contain the strings listed in the command

Cleaning up the values so that they match the values of cantons in the Crime dataset

In [ ]:
#cleaning so that we only keep the names of the cantons in their language (first term)
PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.split('/').str[0] #splitting and then keeping only the [0] value
values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #see if it worked
print(values)

In [ ]:
#stripping values of the whitespaces and the "-"

PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.replace('-', '', regex=False) #regex=False --> look at exact character

PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.strip()

values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #see if it worked
print(values)

In [ ]:
#dropping all rows with "Ohne Angabe" because if we do not get a canton the values are useless to us

PopClean = PopClean[PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] != "Ohne Angabe"]
values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #see if it worked
print(values)

#and renaming the the column name

PopClean = PopClean.rename(columns={"Kanton (-) / Bezirk (>>) / Gemeinde (......)": "Kanton"})

Cleaning the file so that the ages match

In [ ]:
#rename column so that it is easier to work with:
PopClean["Alter"] = PopClean["Alter"].str.strip() #stripping of all whitespaces
PopClean["Alter"] = PopClean["Alter"].str.split(' ').str[0] #only taking first value

#dropping all data that is not 19<=age<=29
PopClean["Alter"] = pd.to_numeric(PopClean["Alter"], errors="coerce")
PopClean = PopClean[PopClean["Alter"] >= 20]
PopClean = PopClean[PopClean["Alter"] <= 29]

#test if it worked:
values = PopClean["Alter"].unique() #see if it worked
print(values)
print(PopClean.columns)
print(PopClean.head())

In [ ]:
#filtering for type of visa
print(PopClean["Bevölkerungstyp"].unique()) #checking what current values are

PopClean = PopClean[PopClean["Bevölkerungstyp"] == "Ständige Wohnbevölkerung"] #including only permanent Stays
print(PopClean["Bevölkerungstyp"].unique())

#dropping collumn "Staatsangehörigkeit" because it is a granular level we don't need
PopClean = PopClean.drop(columns=["Staatsangehörigkeit (Kategorie)"])


print(values)
print(PopClean.columns)
print(PopClean.head())

Renaming cantons so that they will match the other dataset

In [ ]:
#looking exactly at what we have in current working file
values = PopClean["Kanton"].unique() #see if it worked
print(values)
print(PopClean.columns)
print(PopClean.head())

#looking at what Caner did for the Crime file
%cd "/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/Python-Project-HS26-Criminality-and-Minimum-Wage/Group_project_part_caner"
%pwd

crime = pd.read_csv("df_B_20-29_kombiniert_clean.csv")


cvalues = crime["Kanton"].unique() #see if it worked
print(cvalues)
print(crime.columns)
print(crime.head())

#dropping cantons that are not in the Crime dataset and changing the names so that they match
PopClean = PopClean[~PopClean["Kanton"].isin(["Uri", "Schwyz", "Obwalden", "Nidwalden", "Glarus", "Zug", "Schaffhausen", "Appenzell Ausserrhoden", "Appenzell Innerrhoden", "Graubünden", "Ticino", "Jura"])] #.isin checks if canton is in list --> ~ reverses condition

#now renaming values so that they match
PopClean["Kanton"] = PopClean["Kanton"].replace({"BaselLandschaft": "Basel-Landschaft", "BaselStadt": "Basel-Stadt", "Fribourg": "Freiburg", "Genève": "Genf", "Neuchâtel": "Neuenburg", "Vaud": "Waadt", "Valais": "Wallis"})


#see if it worked:
values = PopClean["Kanton"].unique() #see if it worked
print(values)
cvalues = crime["Kanton"].unique() #see if it worked
print(cvalues)


In [ ]:
print(PopClean.head())
print(crime.head())

gender = PopClean["Geschlecht"].unique() #see if it worked
print(gender)

In [ ]:
#grouping for ages across 20-29
PopClean['DATA'] = pd.to_numeric(PopClean['DATA'], errors='coerce')

PopClean = PopClean.groupby(['Jahr', 'Kanton', 'Bevölkerungstyp', 'Geschlecht'], as_index=False)['DATA'].sum()

print(PopClean.head())

Clean_genderless = PopClean[PopClean["Geschlecht"] == "Geschlecht - Total"] #only looking at "genderless" data
Clean_genderless = Clean_genderless.drop(columns=["Geschlecht"])

print(Clean_genderless.head())

In [ ]:
#looking at headers to see how I need to change the structure of the dataset for merging

print(Clean_genderless.head())
print(crime.head())

Relative Population numbers

In [ ]:
Clean_genderless = Clean_genderless[['Kanton', 'Jahr', 'Bevölkerungstyp', 'DATA']]

#create new dataset for swiss data
data_swiss = Clean_genderless[Clean_genderless["Kanton"] == "Schweiz"][["Jahr", "Bevölkerungstyp", "DATA"]].rename(columns={"DATA": "DATA_Schweiz"})

#merging new subset with old dataset
Clean_genderless = Clean_genderless.merge(data_swiss, on=["Jahr", "Bevölkerungstyp"], how="left")
Clean_genderless["Percentage"] = (Clean_genderless["DATA"] / Clean_genderless["DATA_Schweiz"]) * 100


print(Clean_genderless.head())


In [1]:
#converting both "Jahr" to numeric figures
crime["Jahr"] = pd.to_numeric(crime["Jahr"], errors="coerce").astype("Int64")
Clean_genderless["Jahr"] = pd.to_numeric(Clean_genderless["Jahr"], errors="coerce").astype("Int64")

#merging
merged_df = pd.merge(Clean_genderless, crime, on=["Kanton", "Jahr"], how="inner") #inner only keeping exact matches of combinations

print(merged_df.head())

NameError: name 'pd' is not defined

SAVING WHAT I DID IN THE CLEAN DATA FILE

In [ ]:
%cd "/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/Python-Project-HS26-Criminality-and-Minimum-Wage/Data_cleaning_Annette"
%pwd
PopClean.to_csv("Population_clean.csv", index=False)
Clean_genderless.to_csv("Population_clean_without_gender.csv", index=False)
merged_df.to_csv("Crime_Pop_merged.csv", index=False)